In [99]:
import requests
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [100]:
#create tool

@tool
def get_conversion_factor(base_currency:str, target_currency:str) ->float:
    '''
        This function fetches the currency conversion factor between a given base currency and a target currency
    '''

    url = f"https://v6.exchangerate-api.com/v6/b3c34f5883b3792660eb7e39/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value:int, conversion_rate:Annotated[float, InjectedToolArg]) ->float:
    '''Given a currency conversion rate this function calculates the target currency value from a given base currency value.'''
    
    return base_currency_value * conversion_rate

In [101]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1757462401,
 'time_last_update_utc': 'Wed, 10 Sep 2025 00:00:01 +0000',
 'time_next_update_unix': 1757548801,
 'time_next_update_utc': 'Thu, 11 Sep 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 88.2133}

In [102]:
#binding 

model = ChatOpenAI(model='gpt-4o-mini')
model_with_tools = model.bind_tools([get_conversion_factor, convert])

In [103]:
messages = [HumanMessage("What is the conversion factor betweeen USD and INR, and based on that can you convert 10 USD to INR?")]

In [104]:
messages

[HumanMessage(content='What is the conversion factor betweeen USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={})]

In [105]:
ai_message = model_with_tools.invoke(messages)

In [106]:
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_slTOja97mdfzHnYc2c8gEj5y', 'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_vprYXBfSRhTa8tjpJ5g2lq4t', 'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 121, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8bda4d3a2c', 'id': 'chatcmpl-CEBrORQ4K7r9NgZSliwmJtQ6K4S6t', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--c903f292-e888-4072-9d3c-aea1d74b8a9d-0', tool_calls=[{'name': 'get_conversion_factor', 'ar

In [107]:
messages.append(ai_message)

In [108]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_slTOja97mdfzHnYc2c8gEj5y',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_vprYXBfSRhTa8tjpJ5g2lq4t',
  'type': 'tool_call'}]

In [109]:
import json 

for tool_call in ai_message.tool_calls:
    #execute the first tool and get the value of the conersion rate
    if tool_call['name'] =='get_conversion_factor':
        tool_message_1 = get_conversion_factor.invoke(tool_call)
        #fetch this conversion rate
        conversion_rate = json.loads(tool_message_1.content)['conversion_rate']
        #append this tool message to message list
        messages.append(tool_message_1)
    
    #exectue the 2nd tool using the conersion rate from tool 1
    if tool_call['name'] =='convert':
        #fetch the current argument
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message_2 = convert.invoke(tool_call)
        messages.append(tool_message_2)

In [110]:
messages

[HumanMessage(content='What is the conversion factor betweeen USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_slTOja97mdfzHnYc2c8gEj5y', 'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_vprYXBfSRhTa8tjpJ5g2lq4t', 'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 121, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8bda4d3a2c', 'id': 'chatcmpl-CEBrORQ4K7r9NgZSliwmJtQ6K4S6t', 'ser

In [112]:
model_with_tools.invoke(messages).content

'The conversion factor between USD and INR is approximately 88.2133. Therefore, 10 USD is equivalent to about 882.13 INR.'